In [659]:
import torch

In [660]:
words = open("names.txt", "r").read().splitlines()

In [661]:
chars = sorted(set("".join(words))) # setは順番が不安定だから、sortedして順番を確定させる。

stoi = {s:i+1 for i,s in enumerate(chars)}
stoi["."] = 0
itos = {i:s for s,i in stoi.items()}

In [662]:
block_size = 3
# Xにblock_size個の連続文字列のインデックス
# Yにその後に来る文字列のインデックスを入れる
# emmaの場合 X=[5,13,13] Y=[1]
X,Y = [],[]
for w in words:
# for w in ["shido"]:
    # print(w)
    context = [0]*block_size
    for ch in w + ".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        # print("".join(itos[i] for i in context), "-----→", itos[ix])
        context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)

In [663]:
C = torch.randn(27,2) # アルファベッド全27文字を2次元のベクトルで表現する。最初はランダム。

emb = C[X] # これで、Xに該当するインッデクスをCの中から指定できる

In [664]:
W1 = torch.randn(6,100) # ランダムな重み
b1 = torch.randn(100) # ランダムなバイアス

In [665]:
# embは(32,3,2)になっている。32は全学習データ数,3はblocksize(1データごとの文字数),2は埋め込みベクトルの次元数
# これを(32,6)にして3文字分の埋め込みを連結して1本にし、全結合層W1(6,100)に入力できる形にする
torch.cat((emb[:,0,:],emb[:,1,:],emb[:,2,:]), dim=1).shape # torch.cat()でdim次元目を結合できる

torch.Size([228146, 6])

In [666]:
# torch.unbind(emb,1) は (emb[:,0,:],emb[:,1,:],emb[:,2,:])
torch.cat(torch.unbind(emb,1), dim=1).shape
# しかし、torch.catは新しくメモリを使わなきゃいけないため効率が悪いらしい。だから既存のメモリを使いまわせる.viewメソッドを使う↓

torch.Size([228146, 6])

In [667]:
# emb.view(32,6).shape は torch.cat(torch.unbind(emb,1), dim=1).shape と同じ
# emb(192要素)を一度一列に並べて、(32,6)で指定された形に並び替える。順番としては32分割する→6個ずつの塊になっていたらokという操作手順。ずれていたらエラーが出る。
# print(emb.view(32,6).shape)
# -1と置くと6個ずつの塊を作れるだけ作る、逆算手順になる。次元数は(-1,3,2)だったら3次元。(-1,6)だったら二次元になる。-1して良いのは一つの要素まで。
print(emb.view(-1,6).shape)

torch.Size([228146, 6])


In [668]:
# emb.shape = (32,3,2) だから emb.shape[0] = 32
# 拡張性のために具体的な数値はなるべく書かないようにする。6は重みWの形と合わせる必要があるから書いてよい
emb.view(emb.shape[0],6).shape

torch.Size([228146, 6])

In [669]:
# 一層目
h = torch.tanh(emb.view(emb.shape[0],6) @ W1 + b1) # tanh(X*W+b) 活性化関数はtanh

In [670]:
# 二層目
W2 = torch.randn(100,27) #100は前の層のサイズ,27は予測する確率数(アルファベット27文字の確立を予測する)
b2 = torch.randn(27)

logits = h @ W2 + b2

In [671]:
# softmax
counts = logits.exp()
probs = counts / counts.sum(1, keepdim = True)

# クロスエントロピー損失(logの平均)
loss = -probs[torch.arange(X.shape[0]),Y].log().mean()

import torch.nn.functional as F
loss = F.cross_entropy(logits, Y) #この関数一つで、softmaxとクロスエントロピーの計算を行える。
loss

tensor(15.1792)

まとめると

In [672]:
# data分割
def build_dataset(words):
    block_size = 3
    X,Y = [ ],[]
    for w in words:
        context = [0]*block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X,Y

n1 = int(len(words)*0.8)
n2 = int(len(words)*0.9)

Xtr,Ytr = build_dataset(words[:n1])
Xvl,Yvl = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])

torch.Size([182778, 3]) torch.Size([182778])
torch.Size([22633, 3]) torch.Size([22633])
torch.Size([22735, 3]) torch.Size([22735])


In [673]:
g = torch.Generator().manual_seed(2147483647)

C = torch.randn(27,10, generator=g)
W1 = torch.randn(30,200, generator=g)
b1 = torch.randn(200, generator=g)
W2 = torch.randn(200,27, generator=g)
b2 = torch.randn(27, generator=g)

In [674]:
emb = C[Xtr]
emb.shape

torch.Size([182778, 3, 10])

In [675]:
parametors = [C,W1,b1,W2,b2]
for p in parametors:
    p.requires_grad = True # 上のセルのtorch.randnのとこでフラグ建てても良い

In [676]:
for i in range(5000):
    # minibatch
    #randint(a,b,(c,d))はa～bまでのランダムな整数を生成する。生成数は(c,d)。(c,)だけだったらc個のランダムな整数。
    ix = torch.randint(0,Xtr.shape[0], (64,)) #0～Xのデータ数の中からランダムに32(batch_size)個

    # forward
    emb = C[Xtr[ix]] # (32,3,2)
    h = torch.tanh(emb.view(emb.shape[0],-1) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])

    # backward
    for p in parametors:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if i <1000 else 0.01
    for p in parametors:
        p.data -= lr * p.grad

    print("loss=",loss.item(), "lr=",lr)

loss= 26.899703979492188 lr= 0.1
loss= 22.179506301879883 lr= 0.1
loss= 21.547773361206055 lr= 0.1
loss= 23.2850399017334 lr= 0.1
loss= 19.628772735595703 lr= 0.1
loss= 21.59702491760254 lr= 0.1
loss= 19.307252883911133 lr= 0.1
loss= 20.336153030395508 lr= 0.1
loss= 17.988788604736328 lr= 0.1
loss= 16.943124771118164 lr= 0.1
loss= 15.394157409667969 lr= 0.1
loss= 17.165939331054688 lr= 0.1
loss= 18.19731903076172 lr= 0.1
loss= 14.891263961791992 lr= 0.1
loss= 16.09505271911621 lr= 0.1
loss= 15.00747013092041 lr= 0.1
loss= 15.853217124938965 lr= 0.1
loss= 14.442485809326172 lr= 0.1
loss= 15.606517791748047 lr= 0.1
loss= 14.711381912231445 lr= 0.1
loss= 11.097789764404297 lr= 0.1
loss= 13.775980949401855 lr= 0.1
loss= 12.652807235717773 lr= 0.1
loss= 13.622659683227539 lr= 0.1
loss= 14.2658052444458 lr= 0.1
loss= 12.181234359741211 lr= 0.1
loss= 12.909375190734863 lr= 0.1
loss= 16.230335235595703 lr= 0.1
loss= 12.394433975219727 lr= 0.1
loss= 12.484362602233887 lr= 0.1
loss= 12.752724647

In [677]:
emb = C[Xvl]
h = torch.tanh(emb.view(emb.shape[0],-1) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yvl)
loss

tensor(3.3486, grad_fn=<NllLossBackward0>)

In [678]:
# 隠れ層の次元数
# 埋め込み層の次元数
# 学習率を途中で変える
# 入力文字数を3文字から変える
# バッチサイズを変える

In [679]:
# 文字列生成
for i in range(10):
    out = []
    context = [0]*block_size

    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(emb.shape[0],-1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, 1)
        ix = torch.multinomial(probs, 1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break

    print("".join(itos[i] for i in out))

sir.
giviy.
kiz.
abya.
myoriha.
asziyza.
solly.
janlee.
raxtee.
les.


In [680]:
# 可視化用
# from torchviz import make_dot
# make_dot(loss, params={"C": C, "W1": W1, "b1": b1, "W2": W2, "b2": b2})